# 02 — Mammo-FM classifier

This notebook implements the Mammo-FM arm of the downstream classification study across the same four
training conditions and seeds 17, 42, and 73 used for MaxViT-512. The 12 jobs are independent and
resumable, while their data contracts, update budgets, validation rules, and artifact schemas are
held constant to support an architecture-controlled comparison. The official Mammo-FM archive is
resolved locally in offline mode under its original academic-use terms. Neither that archive nor
derived full-state checkpoints are committed or distributed: authorized fine-tuned checkpoints remain
ignored, private artifacts subject to the Mammo-FM license. The held-out test split is not accessed here.

## 1. Resolve the experimental matrix and immutable policy

The configuration cell locates the repository, loads the canonical classifier protocol, and expands
the `mammofm` architecture into four conditions × three seeds. It attaches the signed dataset variant,
training-policy signature, and isolated output directory to every job. Those identities are included
in checkpoints and completion markers, preventing cross-condition or cross-seed reuse.

In [ ]:
from pathlib import Path
import sys
import os

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

# Execution contract: offline and non-mutating unless deliberately opted into.
# See notebooks/utility/review_mode.py for the flags and the environment overrides.
import review_mode
ALLOW_NETWORK_ACCESS = False
INSTALL_DEPENDENCIES = False
ALLOW_PROCESSED_DOWNLOAD = False
review_mode.activate(
    allow_network=ALLOW_NETWORK_ACCESS,
    allow_dependency_install=INSTALL_DEPENDENCIES,
    allow_processed_download=ALLOW_PROCESSED_DOWNLOAD,
)

from notebooks.utility.classifier_experiment import (
    build_error_case_table,
    configure_environment,
    construct_dataset,
    experiment_configuration,
    load_adapter,
    load_existing_outputs,
    load_prediction_rows,
    plot_calibration,
    plot_source_accounting,
    plot_training_history,
    plot_validation_curves,
    run_validation,
    title_classifier_figure,
)

ARCHITECTURE = 'mammofm'
CONDITIONS = (
    'real_only',
    'real_augmented',
    'real_plus_best_finetuned_positive',
    'real_plus_best_fromscratch_positive',
)
SEEDS = (17, 42, 73)
TRAINING_VERBOSE = True
PROGRESS_EVERY_UPDATES = 25  # use 1 to report every optimizer update
VALIDATION_PROGRESS_EVERY_BATCHES = 10

_CLASSIFIER_GPU = os.environ.get('CLASSIFIER_GPU', 'auto')
configurations = {}
for condition in CONDITIONS:
    configurations[condition] = {}
    for seed in SEEDS:
        job_configuration = experiment_configuration(
            ROOT, ARCHITECTURE, condition, seed, gpu=_CLASSIFIER_GPU
        )
        job_configuration['root'] = str(ROOT)
        configurations[condition][seed] = job_configuration

# Compatibility alias used only by the shared GPU-configuration cell below.
configuration = configurations[CONDITIONS[0]][SEEDS[0]]
{
    condition: {
        seed: {
            'experiment_id': configurations[condition][seed]['experiment_id'],
            'results_dir': configurations[condition][seed]['results_dir'],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 2. Configure a portable and verified compute environment

`CLASSIFIER_GPU` may specify `auto`, a physical index, or a runtime UUID; the default automatically
chooses the host device with the greatest reported memory. The selector is resolved against
`nvidia-smi` before framework initialization and the observed CUDA identity is verified afterward.
Only the runtime result is recorded, so the notebook contains no workstation-specific UUID and fails
clearly if device visibility is ambiguous or the kernel was initialized under a different mask.

In [ ]:
# Mammo-FM uses automatic runtime GPU discovery by default. CLASSIFIER_GPU may be supplied
# externally as a runtime selector; no workstation identifier is stored here.
for condition in CONDITIONS:
    for seed in SEEDS:
        configurations[condition][seed]['gpu'] = _CLASSIFIER_GPU
configuration = configurations[CONDITIONS[0]][SEEDS[0]]
{'pinned_gpu': _CLASSIFIER_GPU}

In [ ]:
environment = configure_environment(configuration)
for condition in CONDITIONS:
    for seed in SEEDS:
        job_configuration = configurations[condition][seed]
        job_configuration['gpu_uuid'] = environment['resolved_uuid']
        job_configuration['gpu_name'] = environment['observed_name']
        job_configuration['gpu_physical_index'] = environment['resolved_physical_index']
environment_for_display = {
    key: ('<runtime-resolved>' if key in {'resolved_uuid', 'CUDA_VISIBLE_DEVICES'} else value)
    for key, value in environment.items()
}
environment_for_display


## 3. Construct the four training datasets

The builder combines canonical processed real images with, depending on the experimental condition,
traditional positive augmentations or the canonical FILTERED positive pool of the selected generator.
The same real validation split is used for every job, and validation rows receive neither training
augmentation nor synthetic substitution. Before building a dataset, the code checks that the synthetic
pool exists, contains the expected number of readable positive images, uses unique paths, and does not
reference validation or test data.

In [ ]:
datasets = {
    condition: {
        seed: construct_dataset(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'train_rows': len(datasets[condition][seed]['train_rows']),
            'validation_rows': len(datasets[condition][seed]['validation_rows']),
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 4. Audit composition, accounting, and patient separation

This section reports label/source accounting and the dataset signature of every condition–seed
dataset. It also enforces disjoint training and validation patient sets and rejects forbidden paths,
missing or altered signed samples, and duplicate identities. The audit makes the effective sample
composition explicit before any model state is created.

In [ ]:
{
    condition: {
        seed: datasets[condition][seed]['audit']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 5. Define the shared training and evaluation primitives

The code defines deterministic seeding, fixed-update iteration, binary metrics, atomic history
writing, actual source accounting, and strict resume compatibility. These mechanisms make the
optimization budget comparable across differently sized training conditions and ensure that a
resumed job preserves both statistical state and an auditable count of real, augmented, and
synthetic samples consumed.

In [ ]:
import csv
import gc
import hashlib
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

from notebooks.utility import classifier_checkpoint_io as checkpoint_io
from notebooks.utility.classifier_protocol import atomic_json


## 6. Train or resume all condition–seed jobs

For an incomplete job, the authorized official Mammo-FM checkpoint is resolved from the local
Hugging Face cache with network access disabled, then the architecture adapter constructs the model
and its declared trainable components. AdamW, binary focal loss, mixed precision when supported,
fixed validation intervals, and the shared scheduler/early-stopping policy mirror the MaxViT arm.
The best state maximizes validation PR-AUC under the registered tie-break, and resumable state is
accepted only at a completed validation boundary. Compatible completed runs are verified and skipped
without loading the foundation model. See [`docs/mammo_fm_license_note.md`](../../docs/mammo_fm_license_note.md)
for the non-redistribution boundary.

In [ ]:
def train_one_job(condition, seed, configuration, dataset):
    """Build a fresh state and complete or resume one condition/seed job."""
    CONDITION = str(condition)
    SEED = int(seed)
    POLICY = configuration['policy']
    RUN_DIR = Path(configuration['results_dir'])
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR = Path(configuration['checkpoint_dir'])
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if PROGRESS_EVERY_UPDATES < 1 or VALIDATION_PROGRESS_EVERY_BATCHES < 1:
        raise ValueError('Verbose progress intervals must be positive integers.')
    SOURCE_ACCOUNTING_FIELDS = (
        'real_negative_seen',
        'real_positive_seen',
        'traditional_augmented_seen',
        'finetuned_synthetic_seen',
        'fromscratch_synthetic_seen',
    )
    HISTORY_FIELDS = (
        'loss', 'auc', 'pr_auc', 'val_loss', 'val_auc', 'val_pr_auc',
        'learning_rate', 'optimizer_steps',
    )


    def seed_everything(seed, device):
        random.seed(seed)
        np.random.seed(seed)
        torch.default_generator.manual_seed(seed)
        if device.type == 'cuda':
            torch.cuda.manual_seed(seed)


    def accounting_field(row):
        source = str(row.get('source', '')).lower()
        if source == 'augmented':
            return 'traditional_augmented_seen'
        if source == 'synthetic':
            family = str(row.get('synthetic_family', '')).lower()
            if family == 'finetuned':
                return 'finetuned_synthetic_seen'
            if family == 'from_scratch':
                return 'fromscratch_synthetic_seen'
            raise ValueError(f"Invalid synthetic family: {family!r}")
        return 'real_positive_seen' if int(row['label']) == 1 else 'real_negative_seen'


    def accounting_metadata(rows):
        return [
            {
                'sample_id': str(row.get('image_id') or row.get('sample_id') or index),
                'source': str(row.get('source', 'unknown')),
                'accounting_field': accounting_field(row),
            }
            for index, row in enumerate(rows)
        ]


    class FixedBatchLoader:
        """Repeat the shuffled loader until one fixed validation block is complete."""

        def __init__(self, loader, batch_count):
            self.loader = loader
            self.batch_count = int(batch_count)

        def __len__(self):
            return self.batch_count

        def __iter__(self):
            emitted = 0
            while emitted < self.batch_count:
                cycle_count = 0
                for batch in self.loader:
                    yield batch
                    emitted += 1
                    cycle_count += 1
                    if emitted >= self.batch_count:
                        return
                if cycle_count == 0:
                    raise RuntimeError('Training loader is empty.')


    def binary_metrics(labels, probabilities):
        try:
            roc_auc = float(roc_auc_score(labels, probabilities))
        except ValueError:
            roc_auc = float('nan')
        try:
            pr_auc = float(average_precision_score(labels, probabilities))
        except ValueError:
            pr_auc = float('nan')
        return {'auc': roc_auc, 'pr_auc': pr_auc}


    def accounting_snapshot():
        return {
            'schema_version': 1,
            'accounting_mode': 'actual',
            **source_counts,
            'total_samples_seen': sum(source_counts.values()),
        }


    def write_history_csv(path, values):
        rows = [
            {
                'epoch': index + 1,
                **{
                    field: values[field][index] if index < len(values[field]) else None
                    for field in HISTORY_FIELDS
                },
            }
            for index in range(max((len(values[field]) for field in HISTORY_FIELDS), default=0))
        ]
        temporary = path.with_name(path.name + f'.tmp.{os.getpid()}')
        with temporary.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=['epoch', *HISTORY_FIELDS])
            writer.writeheader()
            writer.writerows(rows)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)


    def normalized_gpu_uuid(value):
        return str(value or '').strip().lower().removeprefix('gpu-')


    validation_interval = int(POLICY['validation_interval_updates'])
    max_optimizer_updates = int(POLICY['max_optimizer_updates'])
    epochs = min(
        int(POLICY.get('max_epochs_secondary_limit', 60)),
        max(1, math.ceil(max_optimizer_updates / validation_interval)),
    )
    early_stopping_patience = int(POLICY['early_stopping']['patience'])
    completion_limits = {
        'max_optimizer_updates': max_optimizer_updates,
        'max_epochs': epochs,
        'early_stopping_patience': early_stopping_patience,
    }
    expected_checkpoint = {
        'architecture': ARCHITECTURE,
        'experiment_id': configuration['experiment_id'],
        'dataset_variant_id': CONDITION,
        'training_policy': configuration['training_policy_name'],
        'config_signature': configuration['policy_signature'],
        'dataset_signature': dataset['dataset_metadata']['signature'],
        'seed': int(SEED),
    }
    completed_run, completion_source = checkpoint_io.inspect_completed_run(
        RUN_DIR, CHECKPOINT_DIR, expected_checkpoint, completion_limits
    )
    if completed_run is not None:
        if completion_source != checkpoint_io.COMPLETION_NAME:
            atomic_json(checkpoint_io.completion_path(RUN_DIR), completed_run)
        existing_outputs = load_existing_outputs(ROOT, configuration)
        checkpoint_gpu_uuid = completed_run.get('checkpoint_gpu_uuid')
        runtime_gpu_uuid = configuration.get('gpu_uuid')
        gpu_changed = bool(
            checkpoint_gpu_uuid and runtime_gpu_uuid
            and normalized_gpu_uuid(checkpoint_gpu_uuid) != normalized_gpu_uuid(runtime_gpu_uuid)
        )
        print(
            f'SKIP completed run | reason={completed_run["completion_reason"]} | '
            f'step={completed_run["optimizer_updates_completed"]}/'
            f'{max_optimizer_updates}',
            flush=True,
        )
        return {
            'configuration': configuration,
            'dataset': dataset,
            'training_result': {
                'status': 'already_complete',
                'completion_reason': completed_run['completion_reason'],
                'completion_source': completion_source,
                'checkpoint': existing_outputs['checkpoint'],
                'history': existing_outputs['history'],
                'resumed_from': None,
                'optimizer_updates_limit': max_optimizer_updates,
                'optimizer_updates_completed': completed_run[
                    'optimizer_updates_completed'
                ],
                'epochs': epochs,
                'best_epoch': completed_run.get('best_epoch'),
                'source_accounting': existing_outputs['source_accounting'],
                'output_dir': str(RUN_DIR),
                'checkpoint_gpu_uuid': checkpoint_gpu_uuid,
                'runtime_gpu_uuid': runtime_gpu_uuid,
                'gpu_changed': gpu_changed,
            },
        }

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        try:
            torch.cuda.set_device(device)
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
        except Exception as exc:
            raise RuntimeError(
                'CUDA context is not healthy. Restart only this notebook kernel '
                'before resuming the classifier sweep.'
            ) from exc
    seed_everything(SEED, device)
    from notebooks.utility import mammofm_utils as architecture_utils
    from notebooks.utility import maxvit_utils as common_utils

    model = architecture_utils.build_mammofm_model(
        hf_repo=architecture_utils.DEFAULT_HF_REPO,
        checkpoint_name=architecture_utils.DEFAULT_CHECKPOINT_NAME,
        use_local_checkpoint=False,
        local_files_only=True,
    )[0]
    architecture_utils.freeze_backbone_all(model)
    architecture_utils.unfreeze_head(model)
    architecture_utils.unfreeze_last_n_blocks(model, 2)

    train_frame = pd.DataFrame(dataset['train_rows'])
    validation_frame = pd.DataFrame(dataset['validation_rows'])
    batch_size = int(POLICY['physical_batch_size'])
    workers = int(POLICY.get('dataloader_workers', 0))
    base_train_loader = architecture_utils.make_mammofm_dataloader(
        train_frame, 'processed_path', 'label',
        architecture_utils.DEFAULT_MAMMOFM_MEAN,
        architecture_utils.DEFAULT_MAMMOFM_STD,
        architecture_utils.DEFAULT_IMG_SIZE,
        batch_size=batch_size, shuffle=True, augment=True, seed=SEED,
        num_workers=workers, drop_last=False,
        metadata=accounting_metadata(dataset['train_rows']),
    )
    validation_loader = architecture_utils.make_mammofm_dataloader(
        validation_frame, 'processed_path', 'label',
        architecture_utils.DEFAULT_MAMMOFM_MEAN,
        architecture_utils.DEFAULT_MAMMOFM_STD,
        architecture_utils.DEFAULT_IMG_SIZE,
        batch_size=batch_size, shuffle=False, augment=False, seed=SEED,
        num_workers=workers, drop_last=False,
    )

    accumulation_steps = int(POLICY.get('gradient_accumulation_steps', 1))
    train_loader = FixedBatchLoader(
        base_train_loader,
        validation_interval * accumulation_steps,
    )

    model.to(device)
    parameters_to_optimize = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        parameters_to_optimize,
        lr=float(POLICY['training_phases'][0]['learning_rate']),
        weight_decay=float(POLICY.get('weight_decay', 0.0)),
    )
    criterion = common_utils.BinaryFocalLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=float(POLICY['scheduler_params']['factor']),
        patience=int(POLICY['scheduler_params']['patience']),
        min_lr=float(POLICY['scheduler_params']['min_lr']),
    )
    scaler = (
        torch.amp.GradScaler('cuda')
        if bool(POLICY.get('amp')) and device.type == 'cuda'
        else None
    )

    resume, resume_source = checkpoint_io.load_resume_checkpoint(
        CHECKPOINT_DIR, expected_checkpoint
    )
    checkpoint_files_exist = any(CHECKPOINT_DIR.glob('checkpoint_*'))
    if resume is None and resume_source == 'no resume checkpoint' and checkpoint_files_exist:
        raise RuntimeError(
            f'Checkpoint files exist in {CHECKPOINT_DIR}, but none is resumable. '
            'Move the run directory before starting again from zero.'
        )
    if resume is None and resume_source != 'no resume checkpoint':
        raise RuntimeError(f'Corrupt or incompatible resume checkpoint: {resume_source}')

    required_resume_fields = {
        'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict',
        'epoch', 'global_step', 'rng_states', 'source_accounting',
    }
    if resume is not None:
        missing = sorted(required_resume_fields - set(resume))
        if missing:
            raise RuntimeError(f'Incomplete resume checkpoint: {missing}')
        if int(resume.get('batch_index', -1)) != -1:
            raise RuntimeError('Resume is allowed only at a validated epoch boundary.')
        if resume['source_accounting'].get('accounting_mode') != 'actual':
            raise RuntimeError('Resume checkpoint lacks actual source accounting.')

    runtime_gpu_uuid = configuration.get('gpu_uuid')
    checkpoint_gpu_uuid = resume.get('gpu_uuid') if resume else None
    gpu_execution_metadata = {
        'checkpoint_gpu_uuid': checkpoint_gpu_uuid,
        'runtime_gpu_uuid': runtime_gpu_uuid,
        'gpu_changed': bool(
            checkpoint_gpu_uuid
            and runtime_gpu_uuid
            and normalized_gpu_uuid(checkpoint_gpu_uuid) != normalized_gpu_uuid(runtime_gpu_uuid)
        ),
    }

    start_epoch = 1
    global_step = 0
    best_pr_auc = float('-inf')
    best_validation_loss = float('inf')
    best_epoch = None
    early_stopping_wait = 0
    history = {field: [] for field in HISTORY_FIELDS}
    source_counts = {field: 0 for field in SOURCE_ACCOUNTING_FIELDS}

    if resume is not None:
        model.load_state_dict(resume['model_state_dict'], strict=True)
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        scheduler.load_state_dict(resume['scheduler_state_dict'])
        if scaler is not None and resume.get('scaler_state_dict'):
            scaler.load_state_dict(resume['scaler_state_dict'])
        start_epoch = int(resume['epoch'])
        global_step = int(resume['global_step'])
        best_pr_auc = float(resume.get('best_metric', float('-inf')))
        best_validation_loss = float(
            resume.get('best_validation_loss', float('inf'))
        )
        best_epoch = resume.get('best_epoch')
        early_stopping_wait = int(resume.get('early_stopping_counter', 0))
        prior_history = resume.get('history', {})
        history = {
            field: list(prior_history.get(field, [])) for field in HISTORY_FIELDS
        }
        source_counts.update({
            field: int(resume['source_accounting'].get(field, 0))
            for field in SOURCE_ACCOUNTING_FIELDS
        })
        rng_states = resume['rng_states']
        if rng_states.get('python'):
            random.setstate(rng_states['python'])
        if rng_states.get('numpy'):
            np.random.set_state(rng_states['numpy'])
        if rng_states.get('torch') is not None:
            torch.set_rng_state(rng_states['torch'])
        if torch.cuda.is_available() and rng_states.get('torch_cuda'):
            torch.cuda.set_rng_state_all(rng_states['torch_cuda'])
        for state_key in (
            'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict',
            'scaler_state_dict', 'rng_states',
        ):
            resume.pop(state_key, None)
        del prior_history, rng_states

    atomic_json(
        RUN_DIR / 'model_summary.json',
        {
            'architecture': ARCHITECTURE,
            'parameters': sum(parameter.numel() for parameter in model.parameters()),
            'trainable_parameters': sum(
                parameter.numel()
                for parameter in model.parameters()
                if parameter.requires_grad
            ),
            'input_size': POLICY['input_size'],
        },
    )
    warmup_updates = int(POLICY.get('warmup_updates', 0))
    target_learning_rate = float(POLICY['training_phases'][0]['learning_rate'])
    gradient_clip = POLICY.get('gradient_clipping')
    resume_segment_id = hashlib.sha256(os.urandom(16)).hexdigest()[:16]
    session_start_step = global_step
    session_start_time = time.perf_counter()


    def readable_duration(seconds):
        if seconds is None or not math.isfinite(seconds):
            return 'unknown'
        seconds = max(0, int(seconds))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'


    if TRAINING_VERBOSE:
        print(
            f'Starting {ARCHITECTURE} | condition={CONDITION} | seed={SEED} | '
            f'device={device} | resume={resume_source} | '
            f'step={global_step}/{max_optimizer_updates}',
            flush=True,
        )


    def save_training_checkpoint(next_epoch, improved):
        payload = {
            **expected_checkpoint,
            **gpu_execution_metadata,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'epoch': int(next_epoch),
            'batch_index': -1,
            'global_step': int(global_step),
            'checkpoint_metric': 'val_pr_auc',
            'best_metric': float(best_pr_auc),
            'best_validation_loss': float(best_validation_loss),
            'best_epoch': best_epoch,
            'early_stopping_counter': int(early_stopping_wait),
            'history': history,
            'source_accounting': accounting_snapshot(),
            'gpu_uuid': runtime_gpu_uuid,
            'rng_states': {
                'python': random.getstate(),
                'numpy': np.random.get_state(),
                'torch': torch.get_rng_state(),
                'torch_cuda': (
                    torch.cuda.get_rng_state_all()
                    if torch.cuda.is_available()
                    else []
                ),
            },
            'resume_segment_id': resume_segment_id,
        }
        checkpoint_io.save_resume_checkpoint(
            CHECKPOINT_DIR, payload, best=bool(improved)
        )


    completion_reason = checkpoint_io.terminal_reason(
        resume, completion_limits
    )
    if completion_reason is not None:
        print(
            f'Resume checkpoint is already terminal ({completion_reason}); '
            'finalizing artifacts without another training epoch.',
            flush=True,
        )


    for epoch in range(start_epoch, epochs + 1):
        if completion_reason is not None:
            break
        if global_step >= max_optimizer_updates:
            completion_reason = 'max_optimizer_updates'
            break
        if TRAINING_VERBOSE:
            print(
                f'\nEpoch {epoch}/{epochs} started at optimizer step '
                f'{global_step}/{max_optimizer_updates}.',
                flush=True,
            )

        # ---- Training batches: forward, focal loss, backward, accumulation, AdamW.
        model.train()
        common_utils.refreeze_batchnorm(model)
        optimizer.zero_grad(set_to_none=True)
        train_loss_sum = 0.0
        train_seen = 0
        train_labels = []
        train_probabilities = []

        for batch_index, batch in enumerate(train_loader):
            if len(batch) != 3:
                raise RuntimeError(
                    'Training batches must include source-accounting metadata.'
                )
            images, labels, metadata = batch
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=(scaler is not None and device.type == 'cuda'),
            ):
                logits = model(images).squeeze(-1)
                full_loss = criterion(logits, labels)
                loss = full_loss / accumulation_steps

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            for field in metadata['accounting_field']:
                if field not in source_counts:
                    raise RuntimeError(f'Unknown accounting field: {field}')
                source_counts[field] += 1

            train_loss_sum += float(full_loss.detach()) * images.size(0)
            train_seen += images.size(0)
            train_labels.extend(labels.detach().cpu().numpy())
            train_probabilities.extend(
                torch.sigmoid(logits).detach().cpu().numpy()
            )

            optimizer_boundary = (
                (batch_index + 1) % accumulation_steps == 0
                or (batch_index + 1) == len(train_loader)
            )
            if optimizer_boundary:
                next_step = global_step + 1
                if warmup_updates and next_step <= warmup_updates:
                    warmup_lr = target_learning_rate * next_step / warmup_updates
                    for group in optimizer.param_groups:
                        group['lr'] = warmup_lr

                if gradient_clip is not None:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        parameters_to_optimize, float(gradient_clip)
                    )

                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step = next_step

                report_progress = (
                    global_step == session_start_step + 1
                    or global_step % PROGRESS_EVERY_UPDATES == 0
                    or global_step >= max_optimizer_updates
                )
                if TRAINING_VERBOSE and report_progress:
                    elapsed = time.perf_counter() - session_start_time
                    session_updates = max(global_step - session_start_step, 1)
                    seconds_per_update = elapsed / session_updates
                    eta = seconds_per_update * (max_optimizer_updates - global_step)
                    progress = 100.0 * global_step / max_optimizer_updates
                    print(
                        f'  train | epoch {epoch}/{epochs} | '
                        f'batch {batch_index + 1}/{len(train_loader)} | '
                        f'step {global_step}/{max_optimizer_updates} '
                        f'({progress:5.1f}%) | loss={float(full_loss.detach()):.4f} | '
                        f'lr={optimizer.param_groups[0]["lr"]:.3e} | '
                        f'elapsed={readable_duration(elapsed)} | '
                        f'ETA={readable_duration(eta)}',
                        flush=True,
                    )

                if global_step >= max_optimizer_updates:
                    break

        train_scores = binary_metrics(
            np.asarray(train_labels), np.asarray(train_probabilities)
        )
        train_metrics = {
            'loss': train_loss_sum / max(train_seen, 1),
            **train_scores,
        }

        # ---- Full validation pass: no gradients and no parameter updates.
        model.eval()
        validation_loss_sum = 0.0
        validation_seen = 0
        validation_labels = []
        validation_probabilities = []
        if TRAINING_VERBOSE:
            print(f'  validation | epoch {epoch}/{epochs} started', flush=True)
        with torch.no_grad():
            for validation_batch_index, (images, labels) in enumerate(validation_loader):
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images).squeeze(-1)
                validation_loss = criterion(logits, labels)
                validation_loss_sum += (
                    float(validation_loss.detach()) * images.size(0)
                )
                validation_seen += images.size(0)
                validation_labels.extend(labels.cpu().numpy())
                validation_probabilities.extend(
                    torch.sigmoid(logits).cpu().numpy()
                )
                report_validation = (
                    (validation_batch_index + 1) % VALIDATION_PROGRESS_EVERY_BATCHES == 0
                    or (validation_batch_index + 1) == len(validation_loader)
                )
                if TRAINING_VERBOSE and report_validation:
                    print(
                        f'  validation | batch {validation_batch_index + 1}/'
                        f'{len(validation_loader)}',
                        flush=True,
                    )

        validation_scores = binary_metrics(
            np.asarray(validation_labels),
            np.asarray(validation_probabilities),
        )
        validation_metrics = {
            'loss': validation_loss_sum / max(validation_seen, 1),
            **validation_scores,
        }
        if not math.isfinite(validation_metrics['pr_auc']):
            raise RuntimeError('Validation PR-AUC is not finite.')

        # ---- History, checkpoint selection, scheduler, and early stopping.
        history['loss'].append(train_metrics['loss'])
        history['auc'].append(train_metrics['auc'])
        history['pr_auc'].append(train_metrics['pr_auc'])
        history['val_loss'].append(validation_metrics['loss'])
        history['val_auc'].append(validation_metrics['auc'])
        history['val_pr_auc'].append(validation_metrics['pr_auc'])
        history['learning_rate'].append(float(optimizer.param_groups[0]['lr']))
        history['optimizer_steps'].append(int(global_step))

        primary_improved = validation_metrics['pr_auc'] > best_pr_auc
        tied_with_better_loss = (
            np.isclose(
                validation_metrics['pr_auc'],
                best_pr_auc,
                rtol=1e-12,
                atol=1e-12,
            )
            and validation_metrics['loss'] < best_validation_loss
        )
        improved = bool(primary_improved or tied_with_better_loss)
        if improved:
            best_pr_auc = float(validation_metrics['pr_auc'])
            best_validation_loss = float(validation_metrics['loss'])
            best_epoch = int(epoch)
            early_stopping_wait = 0
        else:
            early_stopping_wait += 1

        scheduler.step(validation_metrics['pr_auc'])
        save_training_checkpoint(epoch + 1, improved)

        print(
            f"Epoch {epoch}/{epochs} | step {global_step}/{max_optimizer_updates} | "
            f"loss={train_metrics['loss']:.4f} | "
            f"PR-AUC={train_metrics['pr_auc']:.4f} | "
            f"val_loss={validation_metrics['loss']:.4f} | "
            f"val_PR-AUC={validation_metrics['pr_auc']:.4f} | "
            f"best_epoch={best_epoch}"
        )

        if early_stopping_wait >= early_stopping_patience:
            completion_reason = 'early_stopping'
            print(
                f'Early stopping at epoch {epoch}; '
                f'best validation PR-AUC={best_pr_auc:.4f}.'
            )
            break

    if completion_reason is None:
        completion_reason = checkpoint_io.terminal_reason(
            {
                'early_stopping_counter': early_stopping_wait,
                'global_step': global_step,
                'epoch': epochs + 1,
            },
            completion_limits,
        )
    if completion_reason is None:
        raise RuntimeError('Training loop ended without a terminal condition.')

    best_resume_path = checkpoint_io.resume_checkpoint_path(
        CHECKPOINT_DIR, 'checkpoint_best'
    )
    if not best_resume_path.is_file():
        raise RuntimeError('Training completed without a best validated checkpoint.')
    best_payload = checkpoint_io.read_resume_checkpoint(best_resume_path)
    model.load_state_dict(best_payload['model_state_dict'], strict=True)
    best_epoch = best_payload.get('best_epoch', best_epoch)
    del best_payload

    checkpoint_path = CHECKPOINT_DIR / 'checkpoint_best.pt'
    temporary_checkpoint = checkpoint_path.with_name(
        checkpoint_path.name + f'.tmp.{os.getpid()}'
    )
    torch.save(
        {
            'schema_version': 1,
            'architecture': ARCHITECTURE,
            'model_state_dict': checkpoint_io.to_cpu(model.state_dict()),
        },
        temporary_checkpoint,
    )
    os.replace(temporary_checkpoint, checkpoint_path)

    configuration_payload = {
        key: configuration[key]
        for key in ('architecture', 'condition', 'seed', 'gpu')
    }
    configuration_payload.update({
        'experiment_id': configuration['experiment_id'],
        'training_policy': configuration['training_policy_name'],
        'policy_signature': configuration['policy_signature'],
        'dataset_signature': dataset['dataset_metadata']['signature'],
        'completion_reason': completion_reason,
        'training_budget': {
            'max_optimizer_updates': max_optimizer_updates,
            'checkpoint_metric': POLICY['checkpoint_criterion'],
            'scheduler_monitor': POLICY['scheduler_params']['monitor'],
            'early_stopping_monitor': POLICY['early_stopping']['monitor'],
            'effective_batch_size': int(POLICY['effective_batch_size']),
            'validation_interval': validation_interval,
            'validation_manifest': 'data/processed/metadata/val.csv',
        },
        **gpu_execution_metadata,
    })
    atomic_json(RUN_DIR / 'configuration.json', configuration_payload)
    atomic_json(
        RUN_DIR / 'dataset_summary.json',
        {
            **dataset['audit']['full'],
            'validation_manifest': 'data/processed/metadata/val.csv',
            'validation_signature': dataset['dataset_metadata'].get(
                'validation_signature'
            ),
        },
    )
    write_history_csv(RUN_DIR / 'training_history.csv', history)
    source_accounting = accounting_snapshot()
    atomic_json(RUN_DIR / 'source_accounting.json', source_accounting)
    completion_payload = {
        'schema_version': 1,
        'status': 'complete',
        **expected_checkpoint,
        'completion_reason': completion_reason,
        'optimizer_updates_limit': max_optimizer_updates,
        'optimizer_updates_completed': global_step,
        'next_epoch': len(history['loss']) + 1,
        'best_epoch': best_epoch,
        'early_stopping_counter': early_stopping_wait,
        **gpu_execution_metadata,
        'final_checkpoint': 'checkpoint_best.pt',
        'artifacts': list(checkpoint_io.FINAL_ARTIFACTS),
    }
    atomic_json(checkpoint_io.completion_path(RUN_DIR), completion_payload)

    training_result = {
        'status': 'complete',
        'completion_reason': completion_reason,
        'checkpoint': str(checkpoint_path),
        'history': history,
        'resumed_from': resume_source if resume is not None else None,
        'optimizer_updates_limit': max_optimizer_updates,
        'optimizer_updates_completed': global_step,
        'epochs': epochs,
        'best_epoch': best_epoch,
        'source_accounting': source_accounting,
        'output_dir': str(RUN_DIR),
        **gpu_execution_metadata,
    }
    training_result
    return {
        'configuration': configuration,
        'dataset': dataset,
        'training_result': training_result,
    }


job_runs = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        print(
            f'\n========== {ARCHITECTURE} | {condition} | seed {seed} =========='
        )
        job_runs[condition][seed] = train_one_job(
            condition,
            seed,
            configurations[condition][seed],
            datasets[condition][seed],
        )
        gc.collect()
        if (
            job_runs[condition][seed]['training_result']['status']
            != 'already_complete'
            and torch.cuda.is_available()
        ):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

training_results = {
    condition: {
        seed: job_runs[condition][seed]['training_result']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'optimizer_updates_completed': training_results[condition][seed][
                'optimizer_updates_completed'
            ],
            'checkpoint': training_results[condition][seed]['checkpoint'],
            'resumed_from': training_results[condition][seed]['resumed_from'],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}

## 7. Reconstruct validation learning curves from persisted artifacts

Loss and validation-discrimination trajectories are rendered from each job's saved history. This is
an artifact-only operation: it neither resumes optimization nor changes the selected checkpoint.
Curves support convergence and instability checks but do not substitute for the patient-level
ensemble analysis.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
training_history_figures = {
    condition: {
        seed: (
            title_classifier_figure(
                plot_training_history(existing_outputs_by_job[condition][seed]['history']),
                ARCHITECTURE, condition, seed, 'Training history',
            )
            if existing_outputs_by_job[condition][seed]['history']
            else None
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
source_accounting_figures = {
    condition: {
        seed: (
            title_classifier_figure(
                plot_source_accounting(
                    existing_outputs_by_job[condition][seed]['source_accounting']
                ),
                ARCHITECTURE, condition, seed, 'Samples processed by source',
            )
            if existing_outputs_by_job[condition][seed]['source_accounting']
            else None
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
{
    condition: {
        seed: {
            'training_curves': training_history_figures[condition][seed],
            'source_accounting': existing_outputs_by_job[condition][seed][
                'source_accounting'
            ],
            'source_accounting_figure': source_accounting_figures[condition][seed],
        }
        for seed in SEEDS
    }
    for condition in CONDITIONS
}


## 8. Resolve and verify the selected checkpoint for every job

The checkpoint map is rebuilt from canonical run directories and verified against architecture,
condition, seed, policy, dataset signature, and completion metadata. Only compatible
`checkpoint_best.pt` artifacts are admitted. This prevents a stale or differently licensed model
state from being substituted silently during validation or interpretation.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
checkpoints = {
    condition: {
        seed: existing_outputs_by_job[condition][seed]['checkpoint']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
missing_checkpoint_jobs = [
    (condition, seed)
    for condition in CONDITIONS
    for seed in SEEDS
    if checkpoints[condition][seed] is None
]
if missing_checkpoint_jobs:
    raise RuntimeError(
        f'No trained checkpoint is available for jobs {missing_checkpoint_jobs}.'
    )
checkpoints


## 9. Produce or reload validation predictions

The architecture adapter applies each validation-selected checkpoint to the unchanged real
validation set and writes aligned probability records. Compatible existing predictions are reused;
incompatible or incomplete artifacts trigger recomputation. Image and patient keys remain attached
so downstream aggregation can verify exact alignment across seeds.

In [ ]:
existing_outputs_by_job = {
    condition: {
        seed: load_existing_outputs(ROOT, configurations[condition][seed])
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
checkpoints = {
    condition: {
        seed: existing_outputs_by_job[condition][seed]['checkpoint']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
missing_checkpoint_jobs = [
    (condition, seed)
    for condition in CONDITIONS
    for seed in SEEDS
    if checkpoints[condition][seed] is None
]
if missing_checkpoint_jobs:
    raise RuntimeError(
        f'No trained checkpoint is available for jobs {missing_checkpoint_jobs}.'
    )

validation_results = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        validation_results[condition][seed] = run_validation(
            ROOT,
            configurations[condition][seed],
            datasets[condition][seed],
            checkpoints[condition][seed],
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

validation_rows_by_job = {
    condition: {
        seed: validation_results[condition][seed]['rows']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
validation_metrics_by_job = {
    condition: {
        seed: validation_results[condition][seed]['metrics']
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
validation_metrics_by_job


## 10. Summarize validation discrimination and decision performance

Registered validation metrics are computed from the aligned predictions for all 12 jobs. Checkpoint
choice and decision thresholds are derived exclusively from validation, and results remain stratified
by condition and seed for the ensemble comparison. No test-set estimate enters this stage.

In [ ]:
for condition in CONDITIONS:
    for seed in SEEDS:
        title_classifier_figure(
            plot_validation_curves(
                validation_rows_by_job[condition][seed],
                validation_metrics_by_job[condition][seed]['threshold'],
            ),
            ARCHITECTURE, condition, seed, 'Validation diagnostics',
        )
validation_metrics_by_job


## 11. Assess probability calibration

Reliability diagrams compare predicted probabilities with empirical validation frequencies without
modifying model or threshold selection. Because the number of positive patients is limited, bin-level
departures are descriptive and may be unstable. The figures therefore document calibration behavior
under the study sample rather than claim transportable clinical calibration.

In [ ]:
calibration_figures = {
    condition: {
        seed: title_classifier_figure(
            plot_calibration(validation_rows_by_job[condition][seed]),
            ARCHITECTURE, condition, seed, 'Validation calibration',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
calibration_figures


## 12. Inspect errors and fine-tuned–versus–from-scratch disagreements

The cell creates traceable validation error tables and extracts cases on which the two synthetic-data
conditions disagree most strongly. It is a read-only analysis of frozen predictions and cannot alter
the registered matrix. The resulting cases support qualitative failure analysis, not post hoc model
selection.

In [ ]:
from notebooks.utility.classifier_interpretability import largest_ft_fs_disagreements

error_tables_by_job = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        job_rows = validation_rows_by_job[condition][seed]
        job_metrics = validation_metrics_by_job[condition][seed]
        error_cases = build_error_case_table(job_rows, job_metrics['threshold'])
        error_tables = {
            'false positives': error_cases[
                error_cases.error_type == 'false_positive'
            ],
            'false negatives': error_cases[
                error_cases.error_type == 'false_negative'
            ],
            'highest-confidence correct predictions': error_cases[
                error_cases.error_type == 'correct'
            ],
        }
        ft_rows = load_prediction_rows(
            ROOT / 'results/3_classifiers/seed_runs' / ARCHITECTURE
            / 'real_plus_best_finetuned_positive' / f'seed_{seed}'
            / 'validation_predictions.csv'
        )
        fs_rows = load_prediction_rows(
            ROOT / 'results/3_classifiers/seed_runs' / ARCHITECTURE
            / 'real_plus_best_fromscratch_positive' / f'seed_{seed}'
            / 'validation_predictions.csv'
        )
        error_tables['largest FT-vs-FS disagreements'] = (
            largest_ft_fs_disagreements(ft_rows, fs_rows)
            if ft_rows and fs_rows
            else None
        )
        error_tables_by_job[condition][seed] = error_tables

error_tables_by_job

## 13. Generate deterministic Mammo-FM attribution diagnostics

For deterministically selected validation cases, the best checkpoint is reloaded and Mammo-FM-specific spatial
attribution plus integrated gradients are computed as arrays; reusable PNG overlays are saved under
`results/3_classifiers/figures/interpretability/`. Deterministic selection makes each run
reproducible, although model-specific TP/TN/FP/FN cases may differ. These saliency methods
measure local model sensitivity and must not be interpreted as causal explanations or validated
lesion localization.

In [ ]:
from notebooks.utility.classifier_interpretability import (
    mammofm_attribution,
    integrated_gradients,
    select_deterministic_cases,
)

INTERPRETABILITY_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

interpretability_by_job = {condition: {} for condition in CONDITIONS}
for condition in CONDITIONS:
    for seed in SEEDS:
        interpretability_cases = select_deterministic_cases(
            validation_rows_by_job[condition][seed],
            validation_metrics_by_job[condition][seed]['threshold'],
        )
        if not interpretability_cases:
            raise RuntimeError(
                f'Deterministic validation cases are required for {condition}, seed {seed}.'
            )
        adapter = load_adapter(configurations[condition][seed])
        model = adapter.load_checkpoint(checkpoints[condition][seed]).to(INTERPRETABILITY_DEVICE)

        case_loader = adapter.build_validation_dataloader(
            interpretability_cases, seed=seed
        )
        attribution_root = (
            Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability'
        )
        attribution_root.mkdir(parents=True, exist_ok=True)
        case_index = 0
        for images, _labels in case_loader:
            for image in images:
                category = interpretability_cases[case_index]['category']
                heatmap = mammofm_attribution(model, image.unsqueeze(0).to(INTERPRETABILITY_DEVICE))
                np.save(attribution_root / f'{category}_{case_index}.npy', heatmap)
                np.save(attribution_root / f'{category}_{case_index}_ig.npy',
                        integrated_gradients(model, image.unsqueeze(0).to(INTERPRETABILITY_DEVICE)))
                case_index += 1
        interpretability_by_job[condition][seed] = {
            'method': 'EfficientNet spatial-feature attribution',
            'cases': interpretability_cases,
        }
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

interpretability_by_job


In [ ]:
from notebooks.utility.classifier_interpretability import (
    render_attribution_overlays,
    save_attribution_figure,
)

gradcam_figures = {
    condition: {
        seed: title_classifier_figure(
            render_attribution_overlays(
                interpretability_by_job[condition][seed]['cases'],
                Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability',
            ),
            ARCHITECTURE, condition, seed, 'Grad-CAM attribution',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
for condition in CONDITIONS:
    for seed in SEEDS:
        save_attribution_figure(
            gradcam_figures[condition][seed],
            ROOT / 'results/3_classifiers',
            architecture=ARCHITECTURE, condition=condition, seed=seed, method='gradcam',
        )
gradcam_figures

In [ ]:
ig_figures = {
    condition: {
        seed: title_classifier_figure(
            render_attribution_overlays(
                interpretability_by_job[condition][seed]['cases'],
                Path(configurations[condition][seed]['checkpoint_dir']) / 'interpretability',
                suffix='_ig', method='Integrated Gradients',
            ),
            ARCHITECTURE, condition, seed, 'Integrated Gradients attribution',
        )
        for seed in SEEDS
    }
    for condition in CONDITIONS
}
for condition in CONDITIONS:
    for seed in SEEDS:
        save_attribution_figure(
            ig_figures[condition][seed],
            ROOT / 'results/3_classifiers',
            architecture=ARCHITECTURE, condition=condition, seed=seed,
            method='integrated_gradients',
        )
ig_figures